# Неделя 3 — Признаки и CatBoost

Задание: `docs/week3_baseline.md`

In [1]:
import sys
sys.path.append('..')
from src.features import make_features
from src.validation import evaluate



## 1. Построение признаков по группам

TODO: реализуйте make_features() в src/features.py (платежи, динамика, объём отношений, сигналы боли).

In [2]:
import pandas as pd

from src.validation import split_by_time

dataset = pd.read_parquet(
    "../data/processed/dataset.parquet"
)

train, valid, test = split_by_time(dataset)

feature_cols = [
    col for col in dataset.columns
    if col not in ["client_id", "snapshot", "target"]
]

x_train = train[feature_cols].copy()
y_train = train["target"]

x_valid = valid[feature_cols].copy()
y_valid = valid["target"]

x_test = test[feature_cols].copy()
y_test = test["target"]

print(x_train.shape, x_valid.shape, x_test.shape)

(66253, 24) (35788, 24) (40265, 24)


## 2. Обучение CatBoost

TODO: CatBoostClassifier с early stopping на eval-срезе по времени (не случайном!).

In [3]:
from catboost import CatBoostClassifier, Pool
cat_cols = [
    "segment",
    "product",
    "region"
]

for col in cat_cols:
    x_train[col] = x_train[col].fillna("unknown").astype(str)
    x_valid[col] = x_valid[col].fillna("unknown").astype(str)
    x_test[col] = x_test[col].fillna("unknown").astype(str)

numeric_cols = [
    col for col in feature_cols
    if col not in cat_cols
]

train_medians = x_train[numeric_cols].median()

x_train[numeric_cols] = x_train[numeric_cols].fillna(train_medians)
x_valid[numeric_cols] = x_valid[numeric_cols].fillna(train_medians)
x_test[numeric_cols] = x_test[numeric_cols].fillna(train_medians)


catboost_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    early_stopping_rounds=100
)

catboost_model.fit(
    x_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(x_valid, y_valid),
    use_best_model=True
)



0:	test: 0.8578341	best: 0.8578341 (0)	total: 87.2ms	remaining: 1m 27s
100:	test: 0.8892437	best: 0.8892437 (100)	total: 2.52s	remaining: 22.5s
200:	test: 0.8910114	best: 0.8911073 (169)	total: 4.65s	remaining: 18.5s
300:	test: 0.8922918	best: 0.8922918 (300)	total: 7.44s	remaining: 17.3s
400:	test: 0.8927249	best: 0.8928104 (385)	total: 9.94s	remaining: 14.8s
500:	test: 0.8931464	best: 0.8931601 (495)	total: 12.2s	remaining: 12.1s
600:	test: 0.8929441	best: 0.8931684 (504)	total: 14.3s	remaining: 9.48s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.8931684427
bestIteration = 504

Shrink model to first 505 iterations.


CatBoostClassifier(depth=6, early_stopping_rounds=100, eval_metric='AUC', iterations=1000, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=100)

In [4]:
### Проверка на валид

catboost_valid_score = catboost_model.predict_proba(
    x_valid
)[:, 1]

catboost_valid_metrics = evaluate(
    y_valid,
    catboost_valid_score,
    "CatBoost"
)

--- CatBoost ---
ROC-AUC: 0.8932
PR-AUC: 0.7854
Precision@10%: 0.6923
Lift@10%: 7.48x
Доля оттока: 0.0925


In [5]:
### финальная проверка на тесте
catboost_test_score = catboost_model.predict_proba(
    x_test
)[:, 1]

catboost_test_metrics = evaluate(
    y_test,
    catboost_test_score,
    "CatBoost — final test"
)

--- CatBoost — final test ---
ROC-AUC: 0.9143
PR-AUC: 0.8212
Precision@10%: 0.9170
Lift@10%: 6.65x
Доля оттока: 0.1379


## 3. Итоговая таблица метрик: правило vs логрегрессия vs CatBoost



In [7]:

comparison = pd.DataFrame([
    {
        "Модель": "Правило",
        "ROC-AUC": 0.8618,
        "PR-AUC": 0.6538,
        "Precision@10%": 0.6417,
        "Lift@10%": 6.93,
        "Доля оттока": 0.0925 
        
    },
    {
        "Модель": "Логистическая регрессия",
        "ROC-AUC": 0.8786,
        "PR-AUC": 0.7117,
        "Precision@10%": 0.6453,
        "Lift@10%": 6.97,
        "Доля оттока": 0.0925
    },
    {
        "Модель": "CatBoost",
        "ROC-AUC": catboost_valid_metrics["roc_auc"],
        "PR-AUC": catboost_valid_metrics["pr_auc"],
        "Precision@10%": catboost_valid_metrics["precision_at_10"],
        "Lift@10%": catboost_valid_metrics["lift_at_10"],
        "Доля оттока": catboost_valid_metrics["churn_rate"]
    }
])

comparison.round(4)

,Модель,ROC-AUC,PR-AUC,Precision@10%,Lift@10%,Доля оттока
0,Правило,0.8618,0.6538,0.6417,6.9300,0.0925
1,Логистическая регрессия,0.8786,0.7117,0.6453,6.9700,0.0925
2,CatBoost,0.8932,0.7854,0.6923,7.4805,0.0925


## 4. Вывод на языке бизнеса

CatBoost показал лучший результат среди трёх моделей.  
На финальном тесте в топ-10% клиентов по риску попадает около 66,5% всех будущих уходов. Среди выбранных клиентов 91,7% действительно уходят, lift = 6,65x.